# Publications markdown generator for academicpages

This notebook is a thin wrapper around the maintained BibTeX helpers in `generator_utils.py`.
It generates publication markdown into `_publications/` using the same code path as `publications.py`
and `pubsFromBib.py`.

Notes:
- front matter `date` uses `YYYY-FullMonth` from the BibTeX `month` field
- BibTeX `day` is ignored in front matter
- filenames and permalinks keep the stable numeric `YYYY-MM-01` prefix
- rerunning overwrites matching filenames, but old files with renamed titles or dates are not removed automatically


In [1]:
from pathlib import Path
import sys

candidate_dirs = [Path.cwd(), Path.cwd() / "markdown_generator"]
NOTEBOOK_DIR = next(
    (path for path in candidate_dirs if (path / "generator_utils.py").exists()),
    None,
)

if NOTEBOOK_DIR is None:
    raise FileNotFoundError(
        "Run this notebook from the repository root or the markdown_generator directory."
    )

REPO_ROOT = NOTEBOOK_DIR.parent
OUTPUT_DIR = REPO_ROOT / "_publications"

if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from generator_utils import (
    build_publication_markdown,
    build_publication_record_from_bib_entry,
    clean_value,
    parse_bibtex_entries,
    write_markdown,
)

print(f"Notebook directory: {NOTEBOOK_DIR}")
print(f"Output directory: {OUTPUT_DIR}")


Notebook directory: /Users/danywaller/code/danywaller.github.io/markdown_generator
Output directory: /Users/danywaller/code/danywaller.github.io/_publications


In [2]:
publist = {
    "journal": {
        "file": "output.bib",
        "venuekey": "journal",
        "venue-pretext": "",
        "collection": "publications",
        "category": "manuscripts",
        "permalink_prefix": "/publication/",
    }
}

publist


{'journal': {'file': 'output.bib',
  'venuekey': 'journal',
  'venue-pretext': '',
  'collection': 'publications',
  'category': 'manuscripts',
  'permalink_prefix': '/publication/'}}

In [3]:
for source_name, source_config in publist.items():
    source_path = NOTEBOOK_DIR / source_config["file"]

    if not source_path.exists():
        print(f"Skipped {source_name}: missing {source_path.name}")
        continue

    for entry in parse_bibtex_entries(source_path):
        try:
            record = build_publication_record_from_bib_entry(entry, source_config)
            md_filename, markdown = build_publication_markdown(record)
            destination = write_markdown(OUTPUT_DIR, md_filename, markdown)
            print(f"Wrote {destination.name}")
        except KeyError as error:
            title = clean_value(entry["fields"].get("title")) or entry["key"]
            print(f"Skipped {title}: missing expected field {error}")


Wrote 2022-08-01-Ultraviolet-and-magnetic-perspectives-at-Reiner-Gamma-and-the-implications-for-solar-wind-weathering.md
Wrote 2024-08-01-Science-Product-Pipelines-and-Archive-Architecture-for-the-DART-Mission.md
Wrote 2024-11-01-Calibration-and-In-flight-Performance-of-DARTs-Didymos-Reconnaissance-and-Asteroid-Camera-for-OpNav-DRACO.md
Wrote 2024-06-01-Extended-Silicic-Volcanism-in-the-Gruithuisen-RegionRevisiting-the-Composition-and-Thermophysical-Properties-of-Gruithuisen-Domes-on-the-Moon.md
Wrote 2024-02-01-Achievement-of-the-Planetary-Defense-Investigations-of-the-Double-Asteroid-Redirection-Test-DART-Mission.md
Wrote 2024-01-01-An-Updated-Shape-Model-of-Dimorphos-from-DART-Data.md
Wrote 2024-10-01-Radar-Circular-Polarization-Ratio-of-Near-Earth-Asteroids-Links-to-Spectral-Taxonomy-and-Surface-Processes.md
Wrote 2022-09-01-Shape-Modeling-of-Dimorphos-for-the-Double-Asteroid-Redirection-Test-DART.md
Wrote 2025-02-01-Elliptical-ejecta-of-asteroid-Dimorphos-is-due-to-its-surface-cur

If you change a publication title or date, the generated filename changes too.
The notebook overwrites matching files, but it does not delete older markdown files with obsolete names.
Remove stale files from `_publications/` manually when you intentionally rename or redate an entry.
